<a href="https://colab.research.google.com/github/paolobalasso/DataScienceCourse/blob/main/notebooks/03_demand_eda_multi_product.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 03 · Demand EDA — Multi-Product Hierarchy

**Data Science · Università Cattolica del Sacro Cuore, Milan**
Instructor: *Paolo Balasso*

---

## What you will explore

When demand is a **hierarchy** of products (SKU → Category → Total), problems appear that do not exist for a single SKU:

1. **Aggregation hides SKU errors** — a good Total can hide bad SKUs; we separate *weighting* from *error cancellation* and look at the bias
2. **WAPE vs MAPE on a hierarchy** — same forecast, two verdicts once an intermittent SKU joins the portfolio
3. **Reconciliation** — bottom-up vs top-down, and why a naive forecast cannot tell them apart
4. **Cannibalisation** — a promo on BEV_A lifts A *and* depresses its substitute BEV_C: headline vs net effect

This notebook is the hands-on companion of the Bonus module *Multi-Product Demand Forecasting* in the main course deck.

**Style.** Short readable code, lots of plots. Mini-exercises are *parameter tweaks* — add noise to one SKU, change the top-down shares, change the cannibalisation rate. Predict first, then run.

---

## 0 · Setup

In [ ]:
# Colab: every library below is preinstalled, nothing to install.
# Local Jupyter: uncomment the next line and run it once.
# %pip install -q "statsmodels>=0.14" "scikit-learn>=1.3" matplotlib seaborn pandas numpy

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)   # hide library deprecation chatter only
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_absolute_percentage_error

NAVY, GOLD, INK, MUTED = '#1B263B', '#D4A017', '#0F141E', '#556070'
GREEN, RED = '#2D6E3E', '#B32D2D'
sns.set_style('whitegrid')
plt.rcParams.update({'font.family': 'DejaVu Sans',
                     'axes.edgecolor': MUTED, 'axes.labelcolor': INK})
# Reproducibility: the data generator below uses its own np.random.default_rng(seed=42).
print(f'Setup complete. numpy {np.__version__} · pandas {pd.__version__} · '
      f'statsmodels {statsmodels.__version__}')

## 1 · A small multi-SKU hierarchy

We build 7 SKUs grouped into 2 categories:

| Category    | SKU      | Avg weekly units | Notes               |
|-------------|----------|------------------|----------------------|
| Beverages   | BEV_A   | 100              | high volume          |
| Beverages   | BEV_B   | 30               | low volume           |
| Beverages   | BEV_C   | 50               | substitute of BEV_A |
| Snacks      | SNK_A   | 80               | high volume          |
| Snacks      | SNK_B   | 15               | very low volume      |
| Snacks      | SNK_C   | 25               | strongly seasonal    |
| Snacks      | SNK_D   | 1.5              | intermittent: many zero-sales weeks |

The averages are starting levels: each SKU also has a small upward trend and a yearly wave. SNK_D is a slow mover — weekly counts with a mean of 1.5, so some weeks it sells nothing. Real catalogues are full of SKUs like this.

The total = Beverages + Snacks = sum of all 7 SKUs. We can forecast at any of three **levels**: Total, Category, SKU.

In [ ]:
def make_hierarchy(n_weeks=104, seed=42):
    rng = np.random.default_rng(seed)
    weeks = pd.date_range('2023-01-02', periods=n_weeks, freq='W-MON')

    config = {
        # SKU       (category, baseline, trend/wk, season_amp, noise_sd)
        'BEV_A':   ('Beverages', 100, 0.20, 12,  6),
        'BEV_B':   ('Beverages',  30, 0.05,  5,  3),
        'BEV_C':   ('Beverages',  50, 0.10,  6,  4),
        'SNK_A':   ('Snacks',    80, 0.15, 10,  5),
        'SNK_B':   ('Snacks',    15, 0.02,  3,  2),
        'SNK_C':   ('Snacks',    25, 0.08,  8,  4),
    }
    data = {}
    cat_of = {}
    t = np.arange(n_weeks)
    for sku, (cat, base, trend, amp, sd) in config.items():
        season = amp*np.sin(2*np.pi*t/52 - 1.2)
        noise  = rng.normal(0, sd, n_weeks)
        data[sku]   = (base + trend*t + season + noise).clip(min=0).round(1)
        cat_of[sku] = cat
    # SNK_D: an intermittent slow mover (Poisson counts, mean 1.5 a week, many zero weeks)
    data['SNK_D']   = rng.poisson(1.5, n_weeks).astype(float)
    cat_of['SNK_D'] = 'Snacks'
    df = pd.DataFrame(data, index=weeks)
    return df, cat_of

df, cat_of = make_hierarchy()
print(f'Shape: {df.shape}  ({df.shape[0]} weeks × {df.shape[1]} SKUs)')
print(f"SNK_D sells nothing in {int((df['SNK_D'] == 0).sum())} of {len(df)} weeks.")
print('\nCategory map:'); print(pd.Series(cat_of))
df.head()

In [ ]:
# Build aggregated views: Category-level and Total
cat_df = (df.T.groupby(pd.Series(cat_of)).sum().T)   # Beverages / Snacks
tot_s  = df.sum(axis=1).rename('TOTAL')
print('Category-level head:'); display(cat_df.head())
print('\nTotal series head:'); display(tot_s.head())

In [ ]:
# Visualise the three hierarchy levels
split = df.index[-12]          # first week of the 12-week holdout used from section 2 on
fig, axes = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
# SKU
for sku in df.columns:
    axes[0].plot(df.index, df[sku], lw=0.9, label=sku, alpha=0.85)
axes[0].set_title('Level 3 — SKU', fontweight='bold', color=INK, loc='left')
axes[0].legend(loc='upper left', ncol=7, fontsize=8)
# Category
for col, color in zip(cat_df.columns, [NAVY, GOLD]):
    axes[1].plot(cat_df.index, cat_df[col], color=color, lw=1.4, label=col)
axes[1].set_title('Level 2 — Category', fontweight='bold', color=INK, loc='left')
axes[1].legend(loc='upper left')
# Total
axes[2].plot(tot_s.index, tot_s.values, color=INK, lw=1.6)
axes[2].set_title('Level 1 — Total', fontweight='bold', color=INK, loc='left')
# Mark the holdout: everything right of the dashed line is "the future" in sections 2-5
for ax in axes:
    ax.axvline(split, color=MUTED, ls='--', lw=1)
axes[2].text(split, axes[2].get_ylim()[1], '  holdout (12 weeks) →', va='top', color=MUTED, fontsize=9)
plt.tight_layout(); plt.show()

## 2 · Naive forecasts at each level

Quick baseline: average of the last 4 weeks. Same logic, three levels.

Look at the plot first: the 12-week holdout (right of the dashed line) sits on the **downslope** of the yearly wave, while the last 4 training weeks are still higher. Expect the naive forecast to be too high for most series.

In [ ]:
H = 12                          # forecast horizon: 12 weeks
train = df.iloc[:-H]; test = df.iloc[-H:]
train_cat, test_cat = cat_df.iloc[:-H], cat_df.iloc[-H:]
train_tot, test_tot = tot_s.iloc[:-H],   tot_s.iloc[-H:]

# Naive: forecast = last 4-week average, repeated for H weeks
def last4_avg(series):
    return np.repeat(series.iloc[-4:].mean(), H)

fc_sku = pd.DataFrame({c: last4_avg(train[c]) for c in train.columns}, index=test.index)
fc_cat = pd.DataFrame({c: last4_avg(train_cat[c]) for c in train_cat.columns}, index=test_cat.index)
fc_tot = pd.Series(last4_avg(train_tot), index=test_tot.index, name='TOTAL')

print('SKU forecasts:'); display(fc_sku.head(3))
print('\nCategory forecasts:'); display(fc_cat.head(3))
print('\nTotal forecast (1st 3 wks):'); print(fc_tot.head(3).round(1).to_string())

## 3 · Aggregation hides SKU errors

**Claim from the slides** (*The Mix Problem — Simpson Strikes Back*): a good number at an aggregate level can hide bad SKU forecasts — and the warehouse stocks SKUs.

Below we measure WAPE at each level, plus the **bias** = Σ(forecast − actual) / Σ actual (positive = over-forecast). The gap between "SKU error" and "Total error" mixes two different effects:

- **Weighting.** A simple average gives SNK_D (about 1.4 units a week) the same vote as BEV_A (about 110). The **pooled** SKU WAPE puts every SKU-week in one pot, so each SKU counts by its volume.
- **Cancellation.** When SKUs are added up, an over-forecast on one SKU can offset an under-forecast on another in the same week. That only happens when the errors have **different signs**.

In [ ]:
def wape(actual, forecast):
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.sum(np.abs(actual - forecast)) / np.sum(np.abs(actual))

def bias(actual, forecast):
    # signed error relative to volume: > 0 means over-forecast
    actual, forecast = np.asarray(actual, float), np.asarray(forecast, float)
    return np.sum(forecast - actual) / np.sum(actual)

# SKU level: per SKU, then simple average / median / pooled (volume-weighted)
sku_wapes  = pd.Series({sku: wape(test[sku], fc_sku[sku]) for sku in test.columns})
sku_biases = pd.Series({sku: bias(test[sku], fc_sku[sku]) for sku in test.columns})
pooled_wape = wape(test.values.ravel(), fc_sku.values.ravel())    # every SKU-week in one pot
# Category level (both categories pooled) and Total level
cat_wape   = wape(test_cat.values.ravel(), fc_cat.values.ravel())
total_wape = wape(test_tot, fc_tot)
total_bias = bias(test_tot, fc_tot)

summary = pd.DataFrame({
    'Level': ['SKU (simple avg)', 'SKU (median)', 'SKU (pooled, volume-weighted)',
              'Category (pooled)', 'Total'],
    'WAPE':  [sku_wapes.mean(), sku_wapes.median(), pooled_wape, cat_wape, total_wape],
    'Bias':  [sku_biases.mean(), sku_biases.median(),
              bias(test.values.ravel(), fc_sku.values.ravel()),
              bias(test_cat.values.ravel(), fc_cat.values.ravel()), total_bias],
}).round(4)
print('Per-SKU WAPE and bias:')
print(pd.DataFrame({'WAPE': sku_wapes, 'Bias': sku_biases}).round(3).to_string())
print('\nLevel comparison:'); display(summary)

share_over = ((fc_sku - test) > 0).values.mean()
weeks_over = int((fc_tot > test_tot).sum())
print(f'Weighting:    SKU simple avg {sku_wapes.mean():.3f} -> SKU pooled {pooled_wape:.3f}'
      f'   (worst SKU: {sku_wapes.idxmax()} at {sku_wapes.max():.3f})')
print(f'Cancellation: SKU pooled {pooled_wape:.3f} -> Total {total_wape:.3f}')
print(f'Bias floor:   Total WAPE can never go below |Total bias| = {abs(total_bias):.3f}')
print(f'{share_over:.0%} of SKU-weeks are over-forecasts; the Total is over-forecast in {weeks_over} of {H} weeks.')
if total_wape - abs(total_bias) < 0.005:
    print('-> Nothing is left to cancel at Total level: the Total error is the bias shared by the SKUs.')

**Read-out (default settings).**
- The Total WAPE is about 8%, while single SKUs go up to about 15% (SNK_C) and about 77% (SNK_D). A good Total hides bad SKUs.
- Most of the gap between the simple-average SKU WAPE (about 20%) and the Total is **weighting**: the pooled SKU WAPE is already about 10%. SNK_D alone drags the simple average up.
- Only the last step, from about 10% to about 8%, is **cancellation**. It is small because the errors mostly share the same sign: the holdout sits on the seasonal downslope, so the naive forecast is too high for about three SKU-weeks in four. The Total is over-forecast in every one of the 12 weeks, so its WAPE equals its bias.
- Bias does not cancel when you aggregate: pooled SKU, pooled Category and Total have exactly the same bias. Report WAPE **and** bias.
- This is not Simpson's paradox. The slide's mix-shift example is about **revenue weights** that change when the product mix changes; here the weights are units and the mix is stable. Same family of problem, though: what you conclude depends on the level and on the weights you look at.

> **Mini-exercise (parametric) — predict first, then run.** In `make_hierarchy()` change one `noise_sd` at a time and re-run the notebook (*Runtime → Run all* on Colab):
> 1. **SNK_B** (a small SKU): from 2 to 10.
> 2. **BEV_A** (about one third of total volume): from 6 to 20 — put SNK_B back to 2 first.
>
> Before each run, write down your guess for that SKU's WAPE, the pooled SKU WAPE and the Total WAPE. Which change moves the Total more, and why? *Hint:* the naive forecast is the average of the last 4 training weeks. What does extra noise in those 4 weeks do to the level you carry forward for 12 weeks?

## 4 · WAPE vs MAPE on a hierarchy — same forecast, two verdicts

MAPE averages percentage errors, so every SKU (and every week) gets the same vote; WAPE weights by volume. On a hierarchy with very different volumes (BEV_A about 110 units a week, SNK_D about 1.4) the two metrics can give very different verdicts on the *same* forecast.

Watch SNK_D: it has zero-sales weeks in the holdout, and a percentage error on a zero actual is undefined. Our hand-written `mape()` does what many hand-written versions do — it silently **drops** those weeks. We also run scikit-learn's `mean_absolute_percentage_error`, which **keeps** them.

In [ ]:
def mape(actual, forecast):
    a = np.asarray(actual, float); f = np.asarray(forecast, float)
    mask = a != 0                      # zero actuals are silently DROPPED
    return np.mean(np.abs((a[mask] - f[mask]) / a[mask]))

per_sku = pd.DataFrame({
    'volume':     train.mean(),
    'zero_weeks': (test == 0).sum(),   # holdout weeks with zero sales
    'MAPE':       {sku: mape(test[sku], fc_sku[sku]) for sku in test.columns},
    'WAPE':       {sku: wape(test[sku], fc_sku[sku]) for sku in test.columns},
}).round(3).sort_values('volume', ascending=False)
display(per_sku)

# Aggregate the two metrics in different ways
agg_mape_simple   = per_sku['MAPE'].mean()
agg_wape_simple   = per_sku['WAPE'].mean()
agg_wape_weighted = wape(test.values.ravel(), fc_sku.values.ravel())
print(f'Simple-average MAPE across SKUs:   {agg_mape_simple:.3f}')
print(f'Simple-average WAPE across SKUs:   {agg_wape_simple:.3f}')
print(f'Volume-weighted WAPE (pool all):   {agg_wape_weighted:.3f}')
print(f'-> Portfolio verdict: MAPE says about {agg_mape_simple:.0%} off, pooled WAPE says about {agg_wape_weighted:.0%} off.')

# The intermittent SKU, three ways
d = 'SNK_D'
n_zero = int((test[d] == 0).sum())
print(f'\n{d}: hand-written mape() uses {H - n_zero} of {H} weeks (drops {n_zero} zero week{"" if n_zero == 1 else "s"}) -> MAPE {mape(test[d], fc_sku[d]):.3f}')
print(f'{d}: WAPE counts every week                                        -> WAPE {wape(test[d], fc_sku[d]):.3f}')
print(f'{d}: scikit-learn MAPE keeps the zero weeks                         -> MAPE {mean_absolute_percentage_error(test[d], fc_sku[d]):.1e}')

In [ ]:
# Visualise: where do MAPE and WAPE disagree?
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(per_sku))
width = 0.35
ax.bar(x - width/2, per_sku['MAPE'], width, label='MAPE (zero weeks dropped)', color=GOLD,
       edgecolor='black', linewidth=0.4)
ax.bar(x + width/2, per_sku['WAPE'], width, label='WAPE', color=NAVY,
       edgecolor='black', linewidth=0.4)
ax.set_xticks(x); ax.set_xticklabels(per_sku.index, rotation=0)
ax.set_ylabel('Error')
ax.set_title('MAPE vs WAPE per SKU — they agree until a slow mover shows up',
             fontweight='bold', color=INK)
ax.legend(loc='upper left')
plt.tight_layout(); plt.show()

gap = (per_sku['MAPE'] - per_sku['WAPE']).abs()
print(f'Biggest disagreement: {gap.idxmax()} (MAPE {per_sku.loc[gap.idxmax(), "MAPE"]:.2f} '
      f'vs WAPE {per_sku.loc[gap.idxmax(), "WAPE"]:.2f}); all other SKUs differ by at most {gap.drop(gap.idxmax()).max():.2f}.')
print('Rule of thumb: report WAPE (plus bias) to the business; read MAPE per SKU only where actuals never get close to zero.')

**Read-out (default settings).**
- For the six regular SKUs, MAPE and WAPE are almost the same (between about 6% and 16%): with no small actuals, the two metrics agree.
- SNK_D changes the picture. The simple-average MAPE across SKUs is about 15%, the pooled WAPE about 10%: same forecast, two verdicts on the portfolio.
- Look at the middle line too: a simple average of the per-SKU WAPEs (about 20%) is no better. The gap comes from the **weights** (every SKU equal vs every unit equal), not from the formula: WAPE does its job only when you pool.
- Inside SNK_D, the hand-written MAPE (about 37%) looks *better* than its WAPE (about 77%), mainly because it skipped the 4 holdout weeks with zero sales — exactly the weeks where the shop sold nothing and the forecast still said about 1.25 units. Keep those weeks, as scikit-learn does, and MAPE explodes to about 10¹⁵: scikit-learn divides by a tiny number instead of zero.
- Two lessons: check how *your* MAPE treats zeros before you trust it, and for slow movers use WAPE, or MASE (mean absolute scaled error, which we discussed with the *Forecast Accuracy Metrics* slide).

## 5 · Reconciliation — bottom-up vs top-down

When you forecast independently at multiple levels, the forecasts will in general **not add up**. Two classical fixes:

- **Bottom-up.** Forecast each SKU; sum them to get Category and Total. *Pros:* respects SKU structure. *Cons:* noisy SKUs propagate up.
- **Top-down.** Forecast the Total; split it back to SKUs by their historical share. *Pros:* uses the most stable signal. *Cons:* loses per-SKU specifics.

We first try both with the naive forecasts from section 2 — and get a surprise. Middle-ground methods (MinT, optimal reconciliation) are out of scope for today.

In [ ]:
# Bottom-up with the naive forecasts: sum SKU forecasts -> Total forecast
fc_total_bu = fc_sku.sum(axis=1)
# Top-down: take the Total forecast, split by each SKU's average historical share
share = train.mean() / train.mean().sum()
fc_sku_td = pd.DataFrame(
    np.outer(fc_tot.values, share.values),
    index=fc_tot.index, columns=train.columns,
)
fc_total_td = fc_sku_td.sum(axis=1)     # re-aggregate the split back to Total

naive_check = pd.DataFrame({
    'method': ['Total direct (naive on Total)', 'Total bottom-up (sum of SKU naive)',
               'Total top-down (split, then re-summed)'],
    'WAPE':   [wape(test_tot, fc_tot), wape(test_tot, fc_total_bu), wape(test_tot, fc_total_td)],
}).round(4)
display(naive_check)
print(f'Largest weekly gap, bottom-up vs direct: {np.max(np.abs(fc_total_bu - fc_tot)):.1e} units')
print(f'Largest weekly gap, top-down  vs direct: {np.max(np.abs(fc_total_td - fc_tot)):.1e} units')

### Why are all three identical?

Stop and think before you read on.

1. **Bottom-up = direct** because the naive forecast is an *average*, and averaging is linear: the average of a sum is the sum of the averages. Adding up 7 naive SKU forecasts gives exactly the naive forecast of the Total.
2. **Top-down = direct**, always and with any model: the shares add up to 1, so splitting the Total and adding it back returns the Total.

So reconciliation only matters when each series gets its **own, non-linear** model that learns its own level and trend. Below we give every series its own **Holt** model: exponential smoothing with a trend, the non-seasonal cousin of the *Holt-Winters Exponential Smoothing* slide.

In [ ]:
def holt_forecast(series, h=H):
    # Holt = exponential smoothing with an additive trend, fitted on one series only
    fit = ExponentialSmoothing(series.values, trend='add',
                               initialization_method='estimated').fit()
    return np.clip(fit.forecast(h), 0, None)          # demand cannot be negative

hw_sku = pd.DataFrame({c: holt_forecast(train[c]) for c in train.columns}, index=test.index)
hw_tot = pd.Series(holt_forecast(train_tot), index=test_tot.index, name='TOTAL')

hw_total_bu = hw_sku.sum(axis=1)                                  # bottom-up
hw_sku_td = pd.DataFrame(np.outer(hw_tot.values, share.values),   # top-down (same shares)
                         index=test.index, columns=train.columns)

print('Holt, first 3 weeks of the holdout:')
print('  Total direct   :', hw_tot.head(3).round(1).to_list())
print('  Total bottom-up:', hw_total_bu.head(3).round(1).to_list())
print('  Actual Total   :', test_tot.head(3).round(1).to_list())
print(f'\nLargest weekly gap, Holt bottom-up vs Holt direct: {np.max(np.abs(hw_total_bu - hw_tot)):.1f} units -> now the methods differ.')

In [ ]:
# Compare bottom-up and top-down with Holt, at Total level and at SKU level
total_tbl = pd.DataFrame({
    'method': ['Direct Total = top-down (Holt on the Total)', 'Bottom-up (sum of 7 SKU Holt models)'],
    'WAPE':   [wape(test_tot, hw_tot), wape(test_tot, hw_total_bu)],
    'Bias':   [bias(test_tot, hw_tot), bias(test_tot, hw_total_bu)],
}).round(4)
print('TOTAL level'); display(total_tbl)

per_sku_rec = pd.DataFrame({
    'WAPE_bottomup': {s: wape(test[s], hw_sku[s]) for s in test.columns},
    'WAPE_topdown':  {s: wape(test[s], hw_sku_td[s]) for s in test.columns},
})
per_sku_rec['winner'] = np.where(per_sku_rec['WAPE_topdown'] < per_sku_rec['WAPE_bottomup'],
                                 'top-down', 'bottom-up')      # compared before rounding
print('\nSKU level'); display(per_sku_rec.round(3))

# Verdicts computed from the numbers above
pool = lambda fc: wape(test.values.ravel(), fc.values.ravel())
bu_pool, td_pool = pool(hw_sku), pool(hw_sku_td)
tot_bu, tot_td = wape(test_tot, hw_total_bu), wape(test_tot, hw_tot)
n_td = int((per_sku_rec['winner'] == 'top-down').sum())
print(f"Total level: {'bottom-up' if tot_bu < tot_td else 'top-down'} wins "
      f"(WAPE bottom-up {tot_bu:.3f} vs top-down {tot_td:.3f})")
print(f"SKU level:   {'top-down' if td_pool < bu_pool else 'bottom-up'} wins on pooled WAPE "
      f"(top-down {td_pool:.3f} vs bottom-up {bu_pool:.3f}); top-down wins {n_td} of {len(per_sku_rec)} SKUs")
print(f'Naive baseline for reference: Total {wape(test_tot, fc_tot):.3f} · SKU pooled {pool(fc_sku):.3f}')

**Read-out (default settings).**
- At **Total** level, bottom-up wins on this holdout (WAPE about 4.3% vs 5.2%). The single Holt model on the Total under-forecasts (bias about −4%). The sum of the SKU models over-forecasts a little (about +3%), and its weekly errors are smaller: the SKU models' errors partly offset each other when added up.
- At **SKU** level, top-down wins on pooled WAPE (about 8.5% vs 10.7%) and on 5 of 7 SKUs. BEV_A, the big SKU with a strong signal of its own, is better with its own model; BEV_C is practically a tie.
- That is the *opposite* of the textbook line "top-down is stable at the aggregate, bottom-up is truer to each SKU". A plausible reading: a Holt model fitted on a small, noisy SKU chases noise in its level and trend, while the share split borrows the cleaner trend of the Total.
- Holt beats the naive baseline at Total level (naive: about 8.3%); at SKU level only the top-down version does (naive pooled: about 10.1%).
- One 12-week holdout is one opinion: change the `seed` in `make_hierarchy()` and the winners can flip. The robust message is the one on the *Hierarchical Forecasting — Top-Down, Bottom-Up, MinT* slide: forecast where the signal is, borrow strength where it is not, and choose by the level the business acts on (replenishment per SKU vs revenue in total).

> **Mini-exercise (parametric).** Replace the historical-share split with a **last-4-week share** (more reactive): `share = train.iloc[-4:].mean() / train.iloc[-4:].mean().sum()` in the naive reconciliation cell. Re-run this section. Does top-down get better or worse at SKU level? Why does it change so little here?

## 6 · Cannibalisation — when a promo on A hurts C

If BEV_A and BEV_C are substitutes, a promo on BEV_A should *lift* BEV_A and *depress* BEV_C. A forecast built SKU by SKU does not know this: each model only sees its own history.

We inject a promotion into the last 12 weeks: BEV_A sells 50% more, and 40% of that extra volume is taken from BEV_C. Then we measure the **net promo effect** on the Beverages category: the gross uplift on A minus the volume cannibalised from C.

*Scope and signs.* There is no price in this data, so we do **not** estimate a cross-elasticity: `substitution_rate` is a share of the uplift, not an elasticity. On signs: the promo *lowers* C's sales, a negative cross-effect on volume. Written as a cross-price elasticity, substitutes have a **positive** sign (A's price goes down, C's quantity goes down too), as on the *Cross-Elasticity of Demand* slide.

In [ ]:
# Inject a promo + cannibalisation event
df_cann = df.copy()
promo_start = -12          # last 12 weeks of the series
# share of BEV_A's extra volume that is taken from BEV_C (0 = none, 1 = all of it)
substitution_rate = 0.40   #@param {type:"slider", min:0, max:1, step:0.05}

uplift = df['BEV_A'].iloc[promo_start:] * 0.50   # +50% lift on BEV_A
df_cann.iloc[promo_start:, df_cann.columns.get_loc('BEV_A')] += uplift.values
df_cann.iloc[promo_start:, df_cann.columns.get_loc('BEV_C')] -= substitution_rate * uplift.values
df_cann = df_cann.clip(lower=0)

# Visualise the impact
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
for ax, sku, color in zip(axes, ['BEV_A','BEV_C'], [GREEN, RED]):
    ax.plot(df.index, df[sku],       color=MUTED, lw=0.9, label='No promo', alpha=0.7)
    ax.plot(df_cann.index, df_cann[sku], color=color, lw=1.3, label='With promo')
    ax.axvspan(df.index[promo_start], df.index[-1], color=GOLD, alpha=0.15, label='Promo window')
    ax.set_title(f'{sku}', fontweight='bold', color=INK, loc='left')
    ax.legend(loc='upper left')
plt.suptitle('Cannibalisation: promo on BEV_A → substitution on BEV_C',
             fontweight='bold', color=INK, y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# Headline vs net. Same 12 weeks, WITHOUT vs WITH the promo (not before/after in time)
win = slice(promo_start, None)
bev = ['BEV_A', 'BEV_B', 'BEV_C']
bev_no_promo   = df.iloc[win][bev].sum().sum()
bev_with_promo = df_cann.iloc[win][bev].sum().sum()
gross_uplift = df_cann['BEV_A'].iloc[win].sum() - df['BEV_A'].iloc[win].sum()
cannibalised = df['BEV_C'].iloc[win].sum() - df_cann['BEV_C'].iloc[win].sum()
bev_c_change = df_cann['BEV_C'].iloc[win].sum() - df['BEV_C'].iloc[win].sum()   # = -cannibalised
net_effect   = bev_with_promo - bev_no_promo

effect = pd.Series({'Gross uplift on BEV_A':   gross_uplift,
                    'Cannibalised from BEV_C': bev_c_change,
                    'Net effect on Beverages': net_effect})
display(effect.round(0).to_frame('units over the promo weeks'))
print(f"BEV_A {gross_uplift / df['BEV_A'].iloc[win].sum():+.0%} · "
      f"BEV_C {bev_c_change / df['BEV_C'].iloc[win].sum():+.0%} · "
      f"cannibalised share of the uplift {cannibalised / gross_uplift:.0%} · "
      f"net = {net_effect / gross_uplift:.0%} of the headline")

# What an SKU-by-SKU forecast misses: a naive BEV_C forecast that knows nothing about the promo
fc_C = np.repeat(df['BEV_C'].iloc[:promo_start].iloc[-4:].mean(), len(df.iloc[win]))
print(f"\nNaive BEV_C forecast, WAPE vs actuals without the promo: {wape(df['BEV_C'].iloc[win], fc_C):.3f}")
print(f"Naive BEV_C forecast, WAPE vs actuals with the promo:    {wape(df_cann['BEV_C'].iloc[win], fc_C):.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(effect.index, effect.values, color=[GREEN, RED, NAVY], edgecolor='black', linewidth=0.4)
ax.axhline(0, color=INK, lw=0.8)
for i, v in enumerate(effect.values):
    ax.text(i, v, f'{v:+.0f}', ha='center', va='bottom' if v >= 0 else 'top',
            fontweight='bold', color=INK)
ax.margins(y=0.15)                     # room for the value labels
ax.set_ylabel('Units over the promo weeks')
ax.set_title('Headline vs net: what the promo really adds to the category',
             fontweight='bold', color=INK)
plt.tight_layout(); plt.show()

# Interpretation that follows the numbers
if cannibalised < 0.5:
    print('Interpretation: no cannibalisation; the whole uplift on BEV_A is extra volume for the category.')
else:
    print(f'Interpretation: the promo lifts BEV_A by {gross_uplift:.0f} units, but {cannibalised:.0f} of them come from BEV_C.')
    print(f'The category gains {net_effect:+.0f} units, {net_effect / gross_uplift:.0%} of the headline. '
          'Judging the promo on BEV_A alone over-states its effect,')
    print('and a BEV_C forecast that ignores the promo is badly wrong in the same weeks.')

> **Mini-exercise (parametric).** Set `substitution_rate` to `0.0` (no cannibalisation), `0.40` (moderate) and `0.95` (near-full substitution), and re-run the two cells above each time. Before you run: is the promo still a success for the category at 0.95? Then write the three-line note you would send to the category manager. (40% is simply the value chosen for this toy example, not a benchmark.)

## 7 · Caveats & next steps

1. **Synthetic data is friendly.** We added one intermittent SKU; real catalogues have thousands of them, plus holiday spikes and new-product launches. None of today's baselines handles those well (intermittent demand has its own methods, e.g. Croston).
2. **One holdout is one opinion.** The bottom-up vs top-down verdict can change with the random seed. In practice, compare methods over several forecast origins (walk-forward validation, *Ensemble & Model Selection* slide).
3. **Reconciliation is active research.** MinT (Wickramasuriya, Athanasopoulos, Hyndman 2019) gives statistically optimal weighted reconciliation — worth knowing.
4. **Cannibalisation needs *causal* analysis**, not just observational. Promo-A and demand-C are co-moving for many reasons (seasonality, weather). Identify substitution with controlled trials or careful synthetic-control methods.
5. **Hierarchy levels carry different KPIs.** Inventory cares about SKU. Revenue cares about Total. Reconciliation choice is a *business* decision, not a *statistical* one.

### Where to go next

- Implement **MinT optimal reconciliation** (Wickramasuriya et al.)
- Promo / price elasticity model on SKU pairs → cross-elasticity matrix (this needs price data, which this notebook does not have)
- Hierarchical Bayesian forecasting (PyMC) — shrinkage across SKUs in a category

### References

- Hyndman & Athanasopoulos, *FPP3* — Ch. 11 (Forecasting hierarchical and grouped time series). otexts.com/fpp3
- Wickramasuriya, S., Athanasopoulos, G., Hyndman, R. (2019). *Optimal Forecast Reconciliation for Hierarchical and Grouped Time Series Through Trace Minimization*. Journal of the American Statistical Association.
- Companion slides: main course deck, Bonus module *Multi-Product Demand Forecasting* (*The Mix Problem — Simpson Strikes Back*, *WAPE vs MAPE — When Weights Flip the Verdict*, *Cross-Product Cannibalisation & Halo*, *Hierarchical Forecasting — Top-Down, Bottom-Up, MinT*), plus *Forecast Accuracy Metrics* and *Cross-Elasticity of Demand*.

---

**End of notebook.** About 20 seconds on a laptop CPU.